# 02 — Model Training & Evaluation

**Machine Learning-Based Network Intrusion Detection**

This notebook walks through model training, comparison,
hyperparameter tuning, and evaluation step by step.

> **Note:** The full pipeline can also be run via `python src/train.py`.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
import pandas as pd
import numpy as np
from src.utils import create_output_directories
create_output_directories()

## 1. Load & Prepare Data

In [ ]:
from src.data_loader import load_dataset, inspect_dataset
from src.preprocessing import identify_columns, prepare_features_target, split_data

df = load_dataset()
_ = inspect_dataset(df)

columns_info = identify_columns(df)
X, y = prepare_features_target(df, columns_info)
X_train, X_test, y_train, y_test = split_data(X, y)

## 2. Train & Compare Models

In [ ]:
from src.train import get_models, train_and_compare

models = get_models()
numerical_cols = columns_info['numerical']
categorical_cols = columns_info['categorical']

comparison_df, trained_pipelines = train_and_compare(
    models, numerical_cols, categorical_cols,
    X_train, X_test, y_train, y_test,
)
comparison_df

## 3. Best Model Selection

In [ ]:
best_model_name = comparison_df.sort_values('F1 (macro)', ascending=False).iloc[0]['Model']
print(f"Best model by macro F1: {best_model_name}")

best_pipeline = trained_pipelines[best_model_name]

## 4. Hyperparameter Tuning

In [ ]:
from src.train import tune_model
from src.preprocessing import build_pipeline
from src.config import RF_PARAM_GRID, HIST_GB_PARAM_GRID

# Tune the best model (example: Random Forest)
if best_model_name in ('Random Forest', 'Extra Trees'):
    param_grid = RF_PARAM_GRID
elif best_model_name == 'HistGradientBoosting':
    param_grid = HIST_GB_PARAM_GRID
else:
    param_grid = None

if param_grid:
    clf = get_models()[best_model_name]
    fresh_pipeline = build_pipeline(numerical_cols, categorical_cols, clf)
    tuned_pipeline = tune_model(
        best_model_name, fresh_pipeline, param_grid,
        X_train, y_train, n_iter=20, cv=3
    )
else:
    tuned_pipeline = best_pipeline
    print('No tuning grid available; using default model.')

## 5. Final Evaluation

In [ ]:
from sklearn.metrics import classification_report
from src.evaluate import plot_confusion_matrix, binary_evaluation, plot_feature_importance
from src.config import FIGURES_DIR, METRICS_DIR

final_pipeline = tuned_pipeline
y_pred = final_pipeline.predict(X_test)

print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# Confusion matrix
plot_confusion_matrix(y_test, y_pred, FIGURES_DIR)

# Binary evaluation
binary_evaluation(y_test, y_pred)

# Feature importance
plot_feature_importance(final_pipeline)

## 6. Save Model

In [ ]:
import joblib
from src.config import MODEL_SAVE_PATH

MODEL_SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(final_pipeline, MODEL_SAVE_PATH)
print(f'Model saved to {MODEL_SAVE_PATH}')

---
**Done!** The trained model is saved and can be used for predictions
via `python src/predict.py <input.csv>` or the Streamlit app (`streamlit run app.py`).